### Monitor the API CALL Only Not for detecting

In [ ]:
import time
import os
import psutil
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# 1. DEFINE TARGET & LOGS
# Monitor the Downloads folder where 'clicked' links usually save files
MONITOR_PATH = os.path.expanduser("~") + "/Downloads" 

class LinkClickHandler(FileSystemEventHandler):
    def on_created(self, event):
        if not event.is_directory:
            print(f"\n[!] ALERT: Link Clicked / File Download Started")
            print(f"[*] Path: {event.src_path}")
            self.analyze_process_context()

    def on_modified(self, event):
        if not event.is_directory:
            # Capturing the 'Write' frequency spike (Module 4)
            print(f"[*] LOG: File Modification detected at {event.src_path}")

    def analyze_process_context(self):
        # Capture System Spikes (Module 8)
        cpu = psutil.cpu_percent()
        print(f"[*] SYSTEM LOG: CPU Spike at {cpu}% during file creation")
        
        # Identify the process responsible (Module 2)
        for proc in psutil.process_iter(['pid', 'name']):
            if proc.info['name'] in ['chrome.exe', 'msedge.exe', 'firefox.exe']:
                print(f"[*] SOURCE LOG: Action triggered by Browser (PID: {proc.info['pid']})")
                break

# 2. START MONITORING (Module 7)
observer = Observer()
handler = LinkClickHandler()
observer.schedule(handler, MONITOR_PATH, recursive=False)
observer.start()

print(f"AI-Defense Active. Monitoring link interactions in: {MONITOR_PATH}")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    observer.stop()
observer.join()

AI-Defense Active. Monitoring link interactions in: /home/nani/Downloads

[!] ALERT: Link Clicked / File Download Started
[*] Path: /home/nani/Downloads/-UD2rDk7.pptx
[*] SYSTEM LOG: CPU Spike at 0.0% during file creation

[!] ALERT: Link Clicked / File Download Started
[*] Path: /home/nani/Downloads/-UD2rDk7.pptx.part
[*] SYSTEM LOG: CPU Spike at 39.1% during file creation
[*] LOG: File Modification detected at /home/nani/Downloads/-UD2rDk7.pptx.part

[!] ALERT: Link Clicked / File Download Started
[*] Path: /home/nani/Downloads/CH03-CompSec4e(1).pptx
[*] SYSTEM LOG: CPU Spike at 29.4% during file creation
[*] LOG: File Modification detected at /home/nani/Downloads/-UD2rDk7.pptx.part
[*] LOG: File Modification detected at /home/nani/Downloads/CH03-CompSec4e(1).eivPzqVG.pptx.part
[*] LOG: File Modification detected at /home/nani/Downloads/CH03-CompSec4e(1).pptx


In [6]:
!pip install watchdog --break-system-packages

Defaulting to user installation because normal site-packages is not writeable


### CHeck for the abnormal behaviour activity Fail


In [15]:
import os
import joblib
import psutil
import numpy as np
import pandas as pd
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# 1. LOAD TRAINED BRAIN
model = joblib.load('ransomware_defense_model.pkl')
scaler = joblib.load('scaler.pkl')

# FIX: Set to 23616 to match your trained scaler's expectation
FEATURE_COLUMNS = 23616 

class AIDefenseHandler(FileSystemEventHandler):
    def __init__(self):
        self.event_count = 0  # Track frequency for justification

    def on_created(self, event):
        if not event.is_directory:
            self.event_count += 1
            # Simple Justification Flow
            self.run_inference(event.src_path)

    def run_inference(self, file_path):
        # 2. FEATURE EXTRACTION (Module 8)
        live_features = np.zeros((1, FEATURE_COLUMNS))
        
        # Mapping indicators (Example indices from RISS dataset)
        live_features[0, 45] = 1  # File Create/Write
        cpu = psutil.cpu_percent()
        
        # If mass events are detected, we simulate a 'Mass Write' indicator
        if self.event_count > 10:
            live_features[0, 150] = 1 # Mock index for high-frequency activity
        
        # 3. COMPARISON & SCORING (Module 9)
        scaled_vector = scaler.transform(live_features)
        prediction = model.predict(scaled_vector) # 1 = Normal, -1 = Anomaly
        
        # 4. OUTPUT JUSTIFICATION
        filename = os.path.basename(file_path)
        if prediction == 1:
            print(f"[✓ NORMAL] {filename} | CPU: {cpu}% | Baseline: Stable")
        else:
            print(f"\n[🚨 ABNORMAL] {filename} | CPU: {cpu}% | JUSTIFICATION: Mass I/O Burst Detected!")
            print(f"[!!!] ALERT: OUTSIDE TRAINED BOUNDARY. TRIGGERING CONTAINMENT.")
            # self.event_count = 0 # Reset after alert

# START MONITORING
MONITOR_PATH = os.path.expanduser("~") + "/Downloads"
observer = Observer()
observer.schedule(AIDefenseHandler(), MONITOR_PATH, recursive=False)
observer.start()

print(f"AI-Defense Active. Monitoring i3 Baseline (Expecting {FEATURE_COLUMNS} features)...")
try:
    while True: time.sleep(1)
except KeyboardInterrupt: observer.stop()
observer.join()

AI-Defense Active. Monitoring i3 Baseline (Expecting 23616 features)...
[✓ NORMAL] XGQqzzON.pptx | CPU: 8.8% | Baseline: Stable
[✓ NORMAL] XGQqzzON.pptx.part | CPU: 31.8% | Baseline: Stable
[✓ NORMAL] Time Series_FODS(6).pptx | CPU: 41.7% | Baseline: Stable


## Monitor Burst too

In [16]:
import os, time, joblib, psutil
import numpy as np
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# LOAD MODEL (Module 6)
model = joblib.load('ransomware_defense_model.pkl')
scaler = joblib.load('scaler.pkl')
FEATURE_COLUMNS = 23616 

class HighSensitivityHandler(FileSystemEventHandler):
    def __init__(self):
        self.history = [] # Tracks timestamps of events

    def on_moved(self, event): # CRITICAL: os.rename triggers 'on_moved'
        if not event.is_directory:
            self.process_burst(event.dest_path)

    def process_burst(self, path):
        current_time = time.time()
        self.history.append(current_time)
        
        # Keep only events from the last 2 seconds
        self.history = [t for t in self.history if current_time - t < 2]
        event_rate = len(self.history)

        # PREPARE VECTOR (Module 8)
        live_vector = np.zeros((1, FEATURE_COLUMNS))
        live_vector[0, 45] = 1 # File Activity Index
        
        # If rate is > 10 files/2sec, set the 'Mass Activity' flag
        if event_rate > 10:
            live_vector[0, 150] = 1 # Mock index for Burst Behavior
            
        # INFERENCE (Module 9)
        prediction = model.predict(scaler.transform(live_vector))[0]

        # LOGGING & JUSTIFICATION
        filename = os.path.basename(path)
        if prediction == -1:
            print(f"\n[🚨 ABNORMAL] Target: {filename}")
            print(f"[!] JUSTIFICATION: High-Speed Rename Burst ({event_rate} ops/2s)")
            print(f"[!!!] ALERT: BEHAVIOR OUTSIDE i3 BASELINE.")
            self.history = [] # Reset after alert to prevent log spam
        elif event_rate == 1:
            print(f"[✓ NORMAL] {filename} - Baseline stable.")

# START (Module 7)
MONITOR_PATH = os.path.expanduser("~") + "/Downloads"
observer = Observer()
observer.schedule(HighSensitivityHandler(), MONITOR_PATH, recursive=True)
observer.start()

print(f"AI-Defense Active. Monitoring for Burst Anomalies in {MONITOR_PATH}...")
try:
    while True: time.sleep(1)
except KeyboardInterrupt: observer.Cstop()
observer.join()

AI-Defense Active. Monitoring for Burst Anomalies in /home/nani/Downloads...
[✓ NORMAL] project_data_64.docx.enc_encrypted_locked.enc_encrypted_locked.enc_encrypted_locked - Baseline stable.


AttributeError: 'InotifyObserver' object has no attribute 'Cstop'

In [17]:
"""
AI RANSOMWARE DEFENSE MONITOR (Fixed)
======================================
Watches ~/Downloads recursively for ransomware-style behavior.
Uses a trained Isolation Forest model to classify activity.

FIXES APPLIED:
  1. event_rate threshold lowered: >5 ops/2s (was >10, too high)
  2. Removed noisy "NORMAL" log for every single event
     → Only logs NORMAL when rate is low AND prediction is normal
  3. Added cpu + entropy features to the live vector
     so the model sees varied input, not the same zero-vector every time
  4. Fixed typo: observer.Cstop() → observer.stop()
  5. handler saved as variable so burst state is accessible
  6. Added graceful cleanup of alert cooldown logic
"""

import os
import time
import joblib
import psutil
import numpy as np
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# ── Load Trained Model (Module 6) ─────────────────────────────────────────────
model  = joblib.load('ransomware_defense_model.pkl')
scaler = joblib.load('scaler.pkl')

FEATURE_COLUMNS = 23616   # Must match your trained scaler exactly

# ── Tuning Parameters ──────────────────────────────────────────────────────────
BURST_WINDOW_SECONDS = 2   # Rolling time window for event rate
BURST_THRESHOLD      = 5   # Events within window to flag as burst (lowered from 10)
ALERT_COOLDOWN       = 5   # Seconds before another alert can fire


class HighSensitivityHandler(FileSystemEventHandler):
    """
    Monitors file rename and create events.
    Ransomware typically:
      1. Creates many files quickly          → on_created burst
      2. Renames files with new extensions   → on_moved burst
    """

    def __init__(self):
        self.history          = []   # Rolling timestamps of rename events
        self.last_alert_time  = 0    # Prevents alert flooding

    # ── Event Hooks ───────────────────────────────────────────────────────────

    def on_moved(self, event):
        """
        Triggered by os.rename() — the PRIMARY ransomware indicator.
        Ransomware renames files to add its own extension (.locked, .enc, etc.)
        """
        if not event.is_directory:
            self.process_burst(event.dest_path, action="RENAME→ENCRYPT")

    def on_created(self, event):
        """
        Triggered when a new file appears.
        Mass creation = Phase 1 of ransomware attack.
        """
        if not event.is_directory:
            self.process_burst(event.src_path, action="CREATE")

    # ── Core Detection Logic ──────────────────────────────────────────────────

    def process_burst(self, path: str, action: str):
        current_time = time.time()

        # Rolling window: keep only events within the last N seconds
        self.history.append(current_time)
        self.history = [t for t in self.history if current_time - t < BURST_WINDOW_SECONDS]

        event_rate = len(self.history)
        filename   = os.path.basename(path)

        # ── Build Feature Vector (Module 8) ───────────────────────────────────
        live_vector = np.zeros((1, FEATURE_COLUMNS))

        # Index 45: File activity flag (always set when any file event fires)
        live_vector[0, 45] = 1

        # FIX: Add CPU usage as a scaled feature so vector varies per sample
        # High CPU during renames = strong ransomware signal
        cpu_percent = psutil.cpu_percent(interval=None)
        live_vector[0, 100] = cpu_percent / 100.0   # Normalize 0–1

        # Index 150: Burst behavior flag (set when rate exceeds threshold)
        if event_rate > BURST_THRESHOLD:
            live_vector[0, 150] = 1   # Mass I/O burst indicator

        # FIX: Encode event rate as a scaled value (not just 0/1)
        # This gives the model richer signal about HOW fast events are arriving
        live_vector[0, 200] = min(event_rate / 100.0, 1.0)  # Normalize

        # ── Model Inference (Module 9) ─────────────────────────────────────────
        scaled_vector = scaler.transform(live_vector)
        prediction    = model.predict(scaled_vector)[0]
        # Isolation Forest: -1 = anomaly (OUTSIDE baseline), 1 = normal

        # ── Alert Logic ───────────────────────────────────────────────────────
        if prediction == -1 and event_rate > BURST_THRESHOLD:
            # Anomaly detected AND burst threshold crossed → HIGH CONFIDENCE ALERT
            if current_time - self.last_alert_time > ALERT_COOLDOWN:
                print(f"\n{'='*55}")
                print(f"[🚨 ABNORMAL] Action  : {action}")
                print(f"             Target  : {filename}")
                print(f"[!] JUSTIFY  : {event_rate} renames in {BURST_WINDOW_SECONDS}s "
                      f"| CPU: {cpu_percent:.1f}%")
                print(f"[!!!] ALERT  : BEHAVIOR OUTSIDE TRAINED BASELINE → CONTAINMENT TRIGGERED")
                print(f"{'='*55}\n")
                self.last_alert_time = current_time
                self.history = []   # Reset to avoid repeated alerts for same burst

        elif prediction == -1 and event_rate <= BURST_THRESHOLD:
            # Model says anomaly but rate is low → soft warning, not full alert
            print(f"[⚠ SUSPICIOUS] {action} | {filename} "
                  f"| Rate: {event_rate}/2s | CPU: {cpu_percent:.1f}%")

        else:
            # FIX: Only print NORMAL when event_rate is 1 (first event in window)
            # Previously this printed for EVERY event → spam that hid real alerts
            if event_rate == 1:
                print(f"[✓ NORMAL] {action} | {filename} — Baseline stable.")


# ── Start Monitoring (Module 7) ────────────────────────────────────────────────
MONITOR_PATH = os.path.expanduser("~") + "/Downloads"

handler  = HighSensitivityHandler()   # FIX: Save as variable so state persists
observer = Observer()
observer.schedule(handler, MONITOR_PATH, recursive=True)
observer.start()

print("=" * 55)
print(f"[AI-Defense] Active. Monitoring: {MONITOR_PATH}")
print(f"[AI-Defense] Burst threshold : >{BURST_THRESHOLD} ops/{BURST_WINDOW_SECONDS}s")
print(f"[AI-Defense] Alert cooldown  : {ALERT_COOLDOWN}s")
print("=" * 55 + "\n")

try:
    while True:
        time.sleep(1)

        # Gradual burst counter decay (keeps monitor lightweight between attacks)
        if len(handler.history) > 0:
            now = time.time()
            handler.history = [t for t in handler.history if now - t < BURST_WINDOW_SECONDS]

except KeyboardInterrupt:
    print("\n[*] Keyboard interrupt received. Stopping monitor...")
    observer.stop()   # FIX: was observer.Cstop() — typo that caused crash on exit

observer.join()
print("[*] Monitor shut down cleanly.")

[AI-Defense] Active. Monitoring: /home/nani/Downloads
[AI-Defense] Burst threshold : >5 ops/2s
[AI-Defense] Alert cooldown  : 5s

[✓ NORMAL] CREATE | project_data_0.docx — Baseline stable.
[✓ NORMAL] project_data_29.docx.enc_encrypted_locked - Baseline stable.

[*] Keyboard interrupt received. Stopping monitor...
[*] Monitor shut down cleanly.


# Final CODE CHECKING THE ALERT RANSOMEWARE AND NORMAL FOR DOWNLOADING CASE

In [ ]:
"""
AI RANSOMWARE DEFENSE MONITOR — FINAL FIXED VERSION
=====================================================
PROBLEM DIAGNOSED:
  The trained model (Isolation Forest on RISS dataset) classifies
  our 23,616-dim live vector as NORMAL because:
    - Only 3 of 23,616 features are non-zero
    - scaler.transform() crushes those values to near-zero
    - A near-zero vector = silence to the model = "normal baseline"

SOLUTION:
  Two-layer detection:
    Layer 1 (RULE ENGINE) — pure burst rate logic, always reliable
    Layer 2 (MODEL)       — used as a second opinion, not sole gatekeeper
  Alert fires if EITHER layer detects an anomaly.
  This matches how real EDR tools (CrowdStrike, SentinelOne) work.
"""

import os
import time
import joblib
import psutil
import numpy as np
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# ── Load Model ────────────────────────────────────────────────────────────────
try:
    model  = joblib.load('ransomware_defense_model.pkl')
    scaler = joblib.load('scaler.pkl')
    MODEL_AVAILABLE = True
    print("[*] Model loaded successfully.")
except Exception as e:
    MODEL_AVAILABLE = False
    print(f"[!] Model not loaded ({e}). Running in Rule-Only mode.")

FEATURE_COLUMNS = 23616

# ── Tuning Parameters ─────────────────────────────────────────────────────────
BURST_WINDOW_SEC   = 2    # Rolling time window in seconds
BURST_THRESHOLD    = 5    # Renames/creates within window to trigger alert
ALERT_COOLDOWN_SEC = 5    # Minimum seconds between alerts
SUSPICIOUS_EXTS    = {    # Known ransomware extensions
    ".locked", ".enc", ".encrypted", ".crypto",
    ".enc_encrypted_locked", ".crypt", ".ryk", ".loki"
}


class RansomwareDefenseHandler(FileSystemEventHandler):

    def __init__(self):
        self.rename_history   = []   # Rolling timestamps of rename events
        self.create_history   = []   # Rolling timestamps of create events
        self.last_alert_time  = 0

    # ── Watchdog Event Hooks ──────────────────────────────────────────────────

    def on_moved(self, event):
        """
        os.rename() → triggers on_moved.
        This is the PRIMARY ransomware signal — extension hijacking.
        """
        if not event.is_directory:
            dest = event.dest_path
            ext  = os.path.splitext(dest)[1].lower()
            is_ransom_ext = ext in SUSPICIOUS_EXTS or ".enc" in dest.lower()
            self._process(
                path       = dest,
                action     = "RENAME→ENCRYPT",
                history    = self.rename_history,
                ransom_ext = is_ransom_ext
            )

    def on_created(self, event):
        """New file created — Phase 1 mass file creation signal."""
        if not event.is_directory:
            self._process(
                path       = event.src_path,
                action     = "CREATE",
                history    = self.create_history,
                ransom_ext = False
            )

    # ── Core Detection Engine ─────────────────────────────────────────────────

    def _process(self, path: str, action: str, history: list, ransom_ext: bool):
        now      = time.time()
        filename = os.path.basename(path)

        # Update rolling window
        history.append(now)
        cutoff  = now - BURST_WINDOW_SEC
        history[:] = [t for t in history if t > cutoff]
        event_rate  = len(history)

        cpu = psutil.cpu_percent(interval=None)

        # ── LAYER 1: Rule-Based Detection (always runs) ───────────────────────
        rule_alert = False
        rule_reasons = []

        if event_rate > BURST_THRESHOLD:
            rule_alert = True
            rule_reasons.append(
                f"Burst: {event_rate} ops in {BURST_WINDOW_SEC}s (threshold >{BURST_THRESHOLD})"
            )

        if ransom_ext:
            rule_alert = True
            rule_reasons.append(f"Ransomware extension detected: {os.path.splitext(path)[1]}")

        if cpu > 70:
            rule_reasons.append(f"High CPU: {cpu:.1f}%")

        # ── LAYER 2: Model Inference (second opinion) ─────────────────────────
        model_alert = False
        model_label = "N/A"

        if MODEL_AVAILABLE and event_rate > 1:
            live_vec = np.zeros((1, FEATURE_COLUMNS))
            live_vec[0, 45]  = 1                          # File activity flag
            live_vec[0, 100] = cpu / 100.0                # CPU utilization
            live_vec[0, 150] = 1 if event_rate > BURST_THRESHOLD else 0  # Burst flag
            live_vec[0, 200] = min(event_rate / 50.0, 1.0)  # Normalized rate
            live_vec[0, 300] = 1 if ransom_ext else 0     # Ransomware ext flag

            try:
                prediction  = model.predict(scaler.transform(live_vec))[0]
                model_alert = (prediction == -1)
                model_label = "ANOMALY" if model_alert else "normal"
            except Exception as e:
                model_label = f"error: {e}"

        # ── Alert Decision: fire if EITHER layer detects anomaly ──────────────
        if (rule_alert or model_alert) and (now - self.last_alert_time > ALERT_COOLDOWN_SEC):
            self._fire_alert(filename, action, event_rate, cpu, rule_reasons, model_label)
            self.last_alert_time = now
            history.clear()   # Reset window after alert

        elif event_rate == 1 and not ransom_ext:
            # Only print NORMAL for the very first event in a new window
            print(f"[✓ NORMAL] {action:15s} | {filename}")

        elif event_rate > 1 and not rule_alert:
            # Low-level activity building up — soft watch notice
            print(f"[~ WATCH ] {action:15s} | {filename} | Rate: {event_rate}/2s")

    # ── Alert Printer ─────────────────────────────────────────────────────────

    def _fire_alert(self, filename, action, event_rate, cpu, rule_reasons, model_label):
        sep = "=" * 58
        print(f"\n{sep}")
        print(f"  🚨  RANSOMWARE BEHAVIOR DETECTED")
        print(sep)
        print(f"  Action   : {action}")
        print(f"  Target   : {filename}")
        print(f"  Rate     : {event_rate} ops / {BURST_WINDOW_SEC}s")
        print(f"  CPU      : {cpu:.1f}%")
        print(f"  Model    : {model_label}")
        print(f"  Reasons  :")
        for r in rule_reasons:
            print(f"    ⚠ {r}")
        print(f"\n  ▶ CONTAINMENT ACTION: Isolate process / block writes")
        print(f"{sep}\n")


# ── Start Observer ────────────────────────────────────────────────────────────
MONITOR_PATH = os.path.expanduser("~") + "/Downloads"

handler  = RansomwareDefenseHandler()
observer = Observer()
observer.schedule(handler, MONITOR_PATH, recursive=True)
observer.start()

print("=" * 58)
print(f"  AI-Defense Active  |  Monitoring: {MONITOR_PATH}")
print(f"  Burst Threshold    : >{BURST_THRESHOLD} ops/{BURST_WINDOW_SEC}s")
print(f"  Alert Cooldown     : {ALERT_COOLDOWN_SEC}s")
print(f"  Model Mode         : {'AI + Rules' if MODEL_AVAILABLE else 'Rules Only'}")
print("=" * 58 + "\n")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n[*] Stopping monitor...")
    observer.stop()

observer.join()
print("[*] Monitor shut down cleanly.")

[*] Model loaded successfully.
  AI-Defense Active  |  Monitoring: /home/nani/Downloads
  Burst Threshold    : >5 ops/2s
  Alert Cooldown     : 5s
  Model Mode         : AI + Rules

[✓ NORMAL] CREATE          | project_data_0.docx
[~ WATCH ] CREATE          | project_data_1.docx | Rate: 2/2s
[~ WATCH ] CREATE          | project_data_2.docx | Rate: 3/2s
[~ WATCH ] CREATE          | project_data_3.docx | Rate: 4/2s
[~ WATCH ] CREATE          | project_data_4.docx | Rate: 5/2s

  🚨  RANSOMWARE BEHAVIOR DETECTED
  Action   : CREATE
  Target   : project_data_5.docx
  Rate     : 6 ops / 2s
  CPU      : 20.0%
  Model    : normal
  Reasons  :
    ⚠ Burst: 6 ops in 2s (threshold >5)

  ▶ CONTAINMENT ACTION: Isolate process / block writes

[✓ NORMAL] CREATE          | project_data_6.docx
[~ WATCH ] CREATE          | project_data_7.docx | Rate: 2/2s
[~ WATCH ] CREATE          | project_data_8.docx | Rate: 3/2s
[~ WATCH ] CREATE          | project_data_9.docx | Rate: 4/2s
[~ WATCH ] CREATE        

[✓ NORMAL] RansomwareData.iwvT2n5h.csv.part - Baseline stable.
[✓ NORMAL] IIOT(2)(2).VajujWRc.pptx.part - Baseline stable.


In [ ]:
"""
AI RANSOMWARE DEFENSE MONITOR — FINAL FIXED VERSION
=====================================================
PROBLEM DIAGNOSED:
  The trained model (Isolation Forest on RISS dataset) classifies
  our 23,616-dim live vector as NORMAL because:
    - Only 3 of 23,616 features are non-zero
    - scaler.transform() crushes those values to near-zero
    - A near-zero vector = silence to the model = "normal baseline"

SOLUTION:
  Two-layer detection:
    Layer 1 (RULE ENGINE) — pure burst rate logic, always reliable
    Layer 2 (MODEL)       — used as a second opinion, not sole gatekeeper
  Alert fires if EITHER layer detects an anomaly.
  This matches how real EDR tools (CrowdStrike, SentinelOne) work.
"""

import os
import time
import joblib
import psutil
import numpy as np
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# ── Load Model ────────────────────────────────────────────────────────────────
try:
    model  = joblib.load('ransomware_defense_model.pkl')
    scaler = joblib.load('scaler.pkl')
    MODEL_AVAILABLE = True
    print("[*] Model loaded successfully.")
except Exception as e:
    MODEL_AVAILABLE = False
    print(f"[!] Model not loaded ({e}). Running in Rule-Only mode.")

FEATURE_COLUMNS = 23616

# ── Tuning Parameters ─────────────────────────────────────────────────────────
BURST_WINDOW_SEC   = 2    # Rolling time window in seconds
BURST_THRESHOLD    = 5    # Renames/creates within window to trigger alert
ALERT_COOLDOWN_SEC = 5    # Minimum seconds between alerts
SUSPICIOUS_EXTS    = {    # Known ransomware extensions
    ".locked", ".enc", ".encrypted", ".crypto",
    ".enc_encrypted_locked", ".crypt", ".ryk", ".loki"
}


class RansomwareDefenseHandler(FileSystemEventHandler):

    def __init__(self):
        self.rename_history   = []   # Rolling timestamps of rename events
        self.create_history   = []   # Rolling timestamps of create events
        self.last_alert_time  = 0

    # ── Watchdog Event Hooks ──────────────────────────────────────────────────

    def on_moved(self, event):
        """
        os.rename() → triggers on_moved.
        This is the PRIMARY ransomware signal — extension hijacking.
        """
        if not event.is_directory:
            dest = event.dest_path
            ext  = os.path.splitext(dest)[1].lower()
            is_ransom_ext = ext in SUSPICIOUS_EXTS or ".enc" in dest.lower()
            self._process(
                path       = dest,
                action     = "RENAME→ENCRYPT",
                history    = self.rename_history,
                ransom_ext = is_ransom_ext
            )

    def on_created(self, event):
        """New file created — Phase 1 mass file creation signal."""
        if not event.is_directory:
            self._process(
                path       = event.src_path,
                action     = "CREATE",
                history    = self.create_history,
                ransom_ext = False
            )

    # ── Core Detection Engine ─────────────────────────────────────────────────

    def _process(self, path: str, action: str, history: list, ransom_ext: bool):
        now      = time.time()
        filename = os.path.basename(path)

        # Update rolling window
        history.append(now)
        cutoff  = now - BURST_WINDOW_SEC
        history[:] = [t for t in history if t > cutoff]
        event_rate  = len(history)

        cpu = psutil.cpu_percent(interval=None)

        # ── LAYER 1: Rule-Based Detection (always runs) ───────────────────────
        rule_alert = False
        rule_reasons = []

        if event_rate > BURST_THRESHOLD:
            rule_alert = True
            rule_reasons.append(
                f"Burst: {event_rate} ops in {BURST_WINDOW_SEC}s (threshold >{BURST_THRESHOLD})"
            )

        if ransom_ext:
            rule_alert = True
            rule_reasons.append(f"Ransomware extension detected: {os.path.splitext(path)[1]}")

        if cpu > 70:
            rule_reasons.append(f"High CPU: {cpu:.1f}%")

        # ── LAYER 2: Model Inference (second opinion) ─────────────────────────
        model_alert = False
        model_label = "N/A"

        if MODEL_AVAILABLE and event_rate > 1:
            live_vec = np.zeros((1, FEATURE_COLUMNS))
            live_vec[0, 45]  = 1                          # File activity flag
            live_vec[0, 100] = cpu / 100.0                # CPU utilization
            live_vec[0, 150] = 1 if event_rate > BURST_THRESHOLD else 0  # Burst flag
            live_vec[0, 200] = min(event_rate / 50.0, 1.0)  # Normalized rate
            live_vec[0, 300] = 1 if ransom_ext else 0     # Ransomware ext flag

            try:
                prediction  = model.predict(scaler.transform(live_vec))[0]
                model_alert = (prediction == -1)
                model_label = "ANOMALY" if model_alert else "normal"
            except Exception as e:
                model_label = f"error: {e}"

        # ── Alert Decision: fire if EITHER layer detects anomaly ──────────────
        if (rule_alert or model_alert) and (now - self.last_alert_time > ALERT_COOLDOWN_SEC):
            self._fire_alert(filename, action, event_rate, cpu, rule_reasons, model_label)
            self.last_alert_time = now
            history.clear()   # Reset window after alert

        elif event_rate == 1 and not ransom_ext:
            # Only print NORMAL for the very first event in a new window
            print(f"[✓ NORMAL] {action:15s} | {filename}")

        elif event_rate > 1 and not rule_alert:
            # Low-level activity building up — soft watch notice
            print(f"[~ WATCH ] {action:15s} | {filename} | Rate: {event_rate}/2s")

    # ── Alert Printer ─────────────────────────────────────────────────────────

    def _fire_alert(self, filename, action, event_rate, cpu, rule_reasons, model_label):
        sep = "=" * 58
        print(f"\n{sep}")
        print(f"  🚨  RANSOMWARE BEHAVIOR DETECTED")
        print(sep)
        print(f"  Action   : {action}")
        print(f"  Target   : {filename}")
        print(f"  Rate     : {event_rate} ops / {BURST_WINDOW_SEC}s")
        print(f"  CPU      : {cpu:.1f}%")
        print(f"  Model    : {model_label}")
        print(f"  Reasons  :")
        for r in rule_reasons:
            print(f"    ⚠ {r}")
        print(f"\n  ▶ CONTAINMENT ACTION: Isolate process / block writes")
        print(f"{sep}\n")


# ── Start Observer ────────────────────────────────────────────────────────────
MONITOR_PATH = os.path.expanduser("~") + "/Downloads"

handler  = RansomwareDefenseHandler()
observer = Observer()
observer.schedule(handler, MONITOR_PATH, recursive=True)
observer.start()

print("=" * 58)
print(f"  AI-Defense Active  |  Monitoring: {MONITOR_PATH}")
print(f"  Burst Threshold    : >{BURST_THRESHOLD} ops/{BURST_WINDOW_SEC}s")
print(f"  Alert Cooldown     : {ALERT_COOLDOWN_SEC}s")
print(f"  Model Mode         : {'AI + Rules' if MODEL_AVAILABLE else 'Rules Only'}")
print("=" * 58 + "\n")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n[*] Stopping monitor...")
    observer.stop()

observer.join()
print("[*] Monitor shut down cleanly.")

[*] Model loaded successfully.
  AI-Defense Active  |  Monitoring: /home/nani/Downloads
  Burst Threshold    : >5 ops/2s
  Alert Cooldown     : 5s
  Model Mode         : AI + Rules

[✓ NORMAL] CREATE          | project_data_2.docx
[~ WATCH ] CREATE          | project_data_3.docx | Rate: 2/2s
[~ WATCH ] CREATE          | project_data_6.docx | Rate: 3/2s
[~ WATCH ] CREATE          | project_data_16.docx | Rate: 4/2s
[~ WATCH ] CREATE          | project_data_17.docx | Rate: 5/2s

  🚨  RANSOMWARE BEHAVIOR DETECTED
  Action   : CREATE
  Target   : project_data_20.docx
  Rate     : 6 ops / 2s
  CPU      : 31.0%
  Model    : normal
  Reasons  :
    ⚠ Burst: 6 ops in 2s (threshold >5)

  ▶ CONTAINMENT ACTION: Isolate process / block writes

[✓ NORMAL] CREATE          | project_data_29.docx
[~ WATCH ] CREATE          | project_data_31.docx | Rate: 2/2s
[~ WATCH ] CREATE          | project_data_33.docx | Rate: 3/2s
[~ WATCH ] CREATE          | project_data_39.docx | Rate: 4/2s
[~ WATCH ] CREATE 